# YOLO11l-seg Training from Scratch (Baseline)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/train_from_scratch.ipynb)

This notebook trains YOLO11l-seg **from scratch** (random initialization) on the TACO dataset.

**Purpose:** Baseline comparison for the DINOv3 distillation approach.

**Model:**
- YOLO11l-seg: 27.7M params, 143.0 GFLOPs

**Dataset:** TACO - Trash Annotations in Context
- 59 classes of litter/trash
- 1,499 images with polygon segmentation masks
- License: CC BY 4.0

## 1. Install Dependencies

In [ ]:
# Install ultralytics
!pip install -q ultralytics gdown

from ultralytics import YOLO
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

In [ ]:
# Additional imports
import os
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Plotting style - clean, no grids
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 11,
})

COLORS = {'blue': '#2563eb', 'red': '#dc2626', 'green': '#059669', 'orange': '#d97706'}

## 2. Hyperparameters

Configure training hyperparameters here. Defaults based on [Ultralytics recommendations](https://docs.ultralytics.com/modes/train/).

In [ ]:
# ============================================================
# HYPERPARAMETERS - Modify these as needed
# ============================================================

# Training
EPOCHS = 50                 # Number of training epochs
BATCH_SIZE = 16             # Batch size (adjust for GPU memory)
LR = 0.01                   # Initial learning rate (Ultralytics default)
LRF = 0.01                  # Final LR = LR * LRF (Ultralytics default)

# Common
IMAGE_SIZE = 640            # Input image size
PATIENCE = 10               # Early stopping patience
NUM_WORKERS = 8             # Data loading workers

# Model
MODEL_CONFIG = "yolo11l-seg.yaml"  # 27.7M params

print("Hyperparameters:")
print(f"  Epochs:      {EPOCHS}")
print(f"  Batch size:  {BATCH_SIZE}")
print(f"  LR:          {LR} -> {LR * LRF} (final)")
print(f"  Image size:  {IMAGE_SIZE}")
print(f"  Num workers: {NUM_WORKERS}")
print(f"  Model:       {MODEL_CONFIG}")

## 3. Dataset Setup

In [ ]:
# Download TACO dataset from Google Drive
import gdown
import zipfile

FILE_ID = "1HZBR53rs_igtr2ZlYqsLrS0uROCcTqzl"
ZIP_PATH = "taco.zip"
DATA_PATH = Path("./data/taco")

if not DATA_PATH.exists():
    print("Downloading TACO dataset from Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('./data')
    print("Dataset ready!")
else:
    print(f"Dataset already exists at {DATA_PATH}")

dataset_path = DATA_PATH
!ls -la ./data/taco/

In [ ]:
# Verify dataset structure
print("Dataset structure:")
print(f"  Root: {dataset_path}")

total_images = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    labels_path = dataset_path / split / 'labels'
    
    if images_path.exists():
        num_images = len(list(images_path.glob('*.jpg'))) + len(list(images_path.glob('*.JPG'))) + len(list(images_path.glob('*.png')))
        total_images += num_images
        print(f"  {split}/images: {num_images} images")
    
    if labels_path.exists():
        num_labels = len(list(labels_path.glob('*.txt')))
        print(f"  {split}/labels: {num_labels} labels")

print(f"\nTotal images: {total_images}")

## 4. Train YOLO11l-seg from Scratch

Training from random initialization (no pretraining).

This is the **baseline** to compare against DINOv3 distillation.

In [ ]:
# Initialize model from scratch (random weights)
model = YOLO(MODEL_CONFIG)  # .yaml = build from scratch

print(f"Model: YOLO11l-seg")
print(f"Task: {model.task}")
print(f"Initialized from: Random weights (scratch)")

In [ ]:
# Training configuration
DATA_YAML = dataset_path / "data.yaml"
OUTPUT_DIR = "./output/from_scratch"
RUN_NAME = "yolo11l_seg_taco_scratch"

print("Training Configuration:")
print(f"  data:     {DATA_YAML}")
print(f"  epochs:   {EPOCHS}")
print(f"  imgsz:    {IMAGE_SIZE}")
print(f"  batch:    {BATCH_SIZE}")
print(f"  lr0:      {LR}")
print(f"  lrf:      {LRF}")
print(f"  workers:  {NUM_WORKERS}")
print(f"  patience: {PATIENCE}")
print(f"  project:  {OUTPUT_DIR}")
print(f"  name:     {RUN_NAME}")

In [ ]:
# Train the model
print("="*60)
print("Starting training from SCRATCH...")
print("="*60)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    lr0=LR,
    lrf=LRF,
    workers=NUM_WORKERS,
    patience=PATIENCE,
    save=True,
    plots=True,
    project=OUTPUT_DIR,
    name=RUN_NAME,
)

print("="*60)
print("Training complete!")
print("="*60)

### 4.1 Training History

In [ ]:
# Plot training history
results_csv = Path(OUTPUT_DIR) / RUN_NAME / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    # Plot losses
    loss_cols = ['train/box_loss', 'train/seg_loss', 'train/cls_loss', 'train/dfl_loss']
    available_loss = [c for c in loss_cols if c in df.columns]
    
    if available_loss:
        fig, axes = plt.subplots(1, len(available_loss), figsize=(4*len(available_loss), 4))
        if len(available_loss) == 1:
            axes = [axes]
        
        for ax, col in zip(axes, available_loss):
            ax.plot(df[col], color=COLORS['red'], linewidth=2)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.set_title(col.split('/')[-1].replace('_', ' ').title(), fontweight='bold')
        
        plt.suptitle('Training from Scratch - Losses', fontweight='bold', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
    
    # Plot mAP curves
    map_cols = ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)']
    available_map = [c for c in map_cols if c in df.columns]
    
    if available_map:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Box mAP
        ax = axes[0]
        if 'metrics/mAP50(B)' in df.columns:
            ax.plot(df['metrics/mAP50(B)'], color=COLORS['blue'], linewidth=2, label='mAP50')
        if 'metrics/mAP50-95(B)' in df.columns:
            ax.plot(df['metrics/mAP50-95(B)'], color=COLORS['orange'], linewidth=2, label='mAP50-95')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('mAP')
        ax.set_title('Box Detection mAP', fontweight='bold')
        ax.legend(frameon=False)
        
        # Mask mAP
        ax = axes[1]
        if 'metrics/mAP50(M)' in df.columns:
            ax.plot(df['metrics/mAP50(M)'], color=COLORS['green'], linewidth=2, label='mAP50')
        if 'metrics/mAP50-95(M)' in df.columns:
            ax.plot(df['metrics/mAP50-95(M)'], color=COLORS['red'], linewidth=2, label='mAP50-95')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('mAP')
        ax.set_title('Segmentation mAP', fontweight='bold')
        ax.legend(frameon=False)
        
        plt.suptitle('Training from Scratch - Validation mAP', fontweight='bold', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print(f"Results not found at {results_csv}")

## 5. Evaluation & Results

In [ ]:
# Validate the model
metrics = model.val()

# Display results nicely
print("\n" + "="*60)
print("       RESULTS: YOLO11l-seg Trained from Scratch")
print("="*60)
print(f"\n  {'Metric':<25} {'Value':>10}")
print(f"  {'-'*35}")
print(f"  {'Box mAP50':<25} {metrics.box.map50:>10.4f}")
print(f"  {'Box mAP50-95':<25} {metrics.box.map:>10.4f}")
print(f"  {'Mask mAP50':<25} {metrics.seg.map50:>10.4f}")
print(f"  {'Mask mAP50-95':<25} {metrics.seg.map:>10.4f}")
print("\n" + "="*60)

In [ ]:
# Visual metrics summary
fig, ax = plt.subplots(figsize=(8, 5))

metrics_names = ['Box\nmAP50', 'Box\nmAP50-95', 'Mask\nmAP50', 'Mask\nmAP50-95']
metrics_values = [metrics.box.map50, metrics.box.map, metrics.seg.map50, metrics.seg.map]
colors = [COLORS['red'], COLORS['red'], COLORS['orange'], COLORS['orange']]
alphas = [1.0, 0.6, 1.0, 0.6]

bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=alphas, edgecolor='white', linewidth=2)

for bar, val in zip(bars, metrics_values):
    ax.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('mAP Score', fontsize=12)
ax.set_title('Training from Scratch Results', fontweight='bold', fontsize=14)
ax.set_ylim(0, max(metrics_values) * 1.2)
plt.tight_layout()
plt.show()

## 6. Inference Demo

In [ ]:
# Run inference on test images with polygon masks
from PIL import Image
import random

test_images_path = dataset_path / "test" / "images"
test_images = list(test_images_path.glob("*.jpg")) + list(test_images_path.glob("*.JPG"))
random.shuffle(test_images)
test_images = test_images[:6]

print(f"Running inference on {len(test_images)} test images...")

In [ ]:
# Visualize predictions with polygon segmentation masks
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, img_path in zip(axes, test_images):
    # Run prediction
    results = model.predict(str(img_path), verbose=False)
    
    # Plot with masks, boxes, and labels
    result_img = results[0].plot(
        masks=True,           # Show segmentation masks
        boxes=True,           # Show bounding boxes
        labels=True,          # Show class labels
        conf=True,            # Show confidence scores
        line_width=2,
    )
    
    ax.imshow(result_img[:, :, ::-1])
    ax.axis('off')
    
    # Count detections
    n_detections = len(results[0].boxes) if results[0].boxes is not None else 0
    ax.set_title(f'{img_path.name[:25]}...\n{n_detections} detections', fontsize=10)

plt.suptitle('YOLO11l-seg (From Scratch) - Predictions with Polygon Masks', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Save & Export

In [ ]:
# Save metrics to JSON
import json

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

metrics_dict = {
    "model": "YOLO11l-seg",
    "training": "from_scratch",
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "lrf": LRF,
        "image_size": IMAGE_SIZE,
    },
    "results": {
        "box_map50": float(metrics.box.map50),
        "box_map": float(metrics.box.map),
        "seg_map50": float(metrics.seg.map50),
        "seg_map": float(metrics.seg.map),
    }
}

with open(f"{OUTPUT_DIR}/metrics_scratch.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

print(f"Metrics saved to: {OUTPUT_DIR}/metrics_scratch.json")

In [ ]:
# Export to ONNX
model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True)
print("Model exported to ONNX format")

In [ ]:
# Download the trained model (for Colab)
try:
    from google.colab import files
    
    best_model_path = Path(OUTPUT_DIR) / RUN_NAME / "weights" / "best.pt"
    
    if best_model_path.exists():
        print(f"Downloading {best_model_path}...")
        files.download(str(best_model_path))
    else:
        print("Best model not found. Available files:")
        for f in Path(OUTPUT_DIR).rglob("*.pt"):
            print(f"  {f}")
except ImportError:
    print("Not running in Colab - skipping download")

## Summary

This notebook trained **YOLO11l-seg from scratch** (random initialization) on the TACO dataset.

**Use this as a baseline** to compare against:
- `dinov3_distillation.ipynb` - YOLO11l-seg pretrained with DINOv3 distillation

**Expected:** The distillation-pretrained model should achieve better mAP scores, especially with limited training data.